In [ ]:
import matplotlib.pyplot as plt
# %matplotlib notebook
# from ipywidgets import *
import numpy as np
import sys
import numpy.typing as npt
import typing as t
import jax.numpy as jnp

In [ ]:
import taurex.log
taurex.log.disableLogging()

In [ ]:
from taurex.cache import OpacityCache,CIACache
from taurex.contributions import AbsorptionContribution, CIAContribution, RayleighContribution
from taurex.data.spectrum.observed import ObservedSpectrum
from taurex.model import TransmissionModel
from taurex.util import clip_native_to_wngrid

from taurex.planet import Planet
from taurex.stellar import BlackbodyStar
from taurex.chemistry import TaurexChemistry, ConstantGas
from taurex.temperature import Isothermal

In [ ]:
xsec_path = "/Users/asweet/Code/research/taurex3/test_files/Input/xsec/xsec_sampled_R15000_0.3-50"
cia_path = "/Users/asweet/Code/research/taurex3/test_files/Input/cia/HITRAN/data"

OpacityCache().clear_cache()
OpacityCache().set_opacity_path(xsec_path)
CIACache().set_cia_path(cia_path)

In [ ]:
planet = Planet(planet_radius=1.0, planet_mass=1.0)
star = BlackbodyStar(temperature=5700.0, radius=1.0)

chemistry = TaurexChemistry(fill_gases=['H2','He'], ratio=0.172)
h2o = ConstantGas('H2O', mix_ratio=1.2e-4)
chemistry.addGas(h2o)
chemistry.addGas(ConstantGas('N2', mix_ratio=3.00739e-9))

In [ ]:
isothermal = Isothermal(T=1500.0)

tm = TransmissionModel(planet=planet,
                       temperature_profile=isothermal,
                       chemistry=chemistry,
                       star=star,
                       atm_min_pressure=1e-0,
                       atm_max_pressure=1e6,
                       nlayers=30)
tm.add_contribution(AbsorptionContribution())
tm.add_contribution(CIAContribution(cia_pairs=['H2-H2','H2-He']))
tm.add_contribution(RayleighContribution())
tm.build()

# adjust gas
tm['H2O'] = 1.2e-4

In [ ]:
obs = ObservedSpectrum('../test_files/quickstart.dat')
obin = obs.create_binner()

In [ ]:
plt.figure()
plt.errorbar(obs.wavelengthGrid, obs.spectrum, obs.errorBar, label='Obs')
plt.plot(obs.wavelengthGrid, obin.bin_model(tm.model(obs.wavenumberGrid))[1], label='TM')
plt.legend()
plt.show()

# Decomposition (DIY)
- numpy (inference)
- clean up
- jax (inference)
- jit and fit
- compare

In [ ]:
for key, val in tm._fitting_parameters.items():
    # * parameter name
    # * parameter name in Latex form
    # * get function
    # * set function
    # * fitting scale
    # * fit as default
    # * fitting boundaries
    get_val = val[2]()
    val_range = val[-1]
    fit_scale = val[4]
    print(key, get_val, val_range, fit_scale)

In [ ]:
tm.contribution_list

In [ ]:
wn_grid = clip_native_to_wngrid(tm.nativeWavenumberGrid, obs.wavenumberGrid)

In [ ]:
star.initialize(wn_grid)

In [ ]:
for contrib in tm.contribution_list:
    contrib.prepare(tm, wn_grid)

In [ ]:
tm.deltaz

In [ ]:
def multi_dot(a, b):
    return np.sum(a*b, axis=0)
    
def compute_intersection_3d(R, h, u, o, c=np.zeros(3), allow_single=False):
    """
    Computes line-sphere intersection for a range of radius offsets
    Will detect if path crosses R and cutoff the point

    Parameters
    ----------
    R: float
        Radius

    h: float or array_like of shape (m)
        height above radius
    
    u: array with shape (3,n)
        normalized direction vector

    o: array with shape (3,n)
        origin vector

    c: vector, optional
        center position of sphere default is (0,0,0)

    Returns
    --------
    solution: array of shape (2,3,m,n)
        Array containing points that intersect the sphere
        First index is always the point closest to origin vector


    """
    if not hasattr(h, '__len__'):
        h = np.array([h])
    else:
        h = np.array(h)
    
    if len(u.shape) == 1:
        u = u.reshape(-1, 1)
        o = o.reshape(-1, 1)
    
    single_dot = multi_dot(u, o)
    dot_res = (single_dot)**2 - (o**2).sum(axis=0)
    delta = dot_res + (R+h[..., None])**2
    solution = np.zeros(shape=(2, 3, h.shape[0], u.shape[1]))
    sol1 = np.zeros(shape=(3, h.shape[0], u.shape[1]))
    sol2 = np.zeros(shape=(3, h.shape[0], u.shape[1]))
    solution[...] = np.nan
    filt = delta > 0
    if allow_single:
        filt = delta >= 0
    if filt.sum() == 0:
        return None
    
    with np.errstate(divide='ignore', invalid='ignore'):

        d1 = -(single_dot) + np.sqrt(delta)
        d1[d1 < 0] = 0.0
        sol1 = o[:, None, :] + d1*u[:, None]

        d2 = -(single_dot) - np.sqrt(delta)
        d2[d2 < 0] = 0.0
        sol2 = o[:, None, :] + d2*u[:, None]
        v1 = np.sum((o[:, None, :]-sol1)**2, axis=0)
        v2 = np.sum((o[:, None, :]-sol2)**2, axis=0)
    #
    #print(v1.shape)
    max_filter = v2 > v1
    #print(max_filter.shape)
    a_filter = max_filter
    b_filter = ~max_filter
    #print(solution.shape)
    if a_filter.sum() > 0:
        solution[0, :, a_filter] = sol1[:, a_filter].T
        solution[1, :, a_filter] = sol2[:, a_filter].T
    if b_filter.sum() > 0:
        solution[0, :, b_filter] = sol2[:, b_filter].T

        solution[1, :, b_filter] = sol1[:, b_filter].T
    
    # Detect planet crossings
    delta = dot_res + (R)**2

    filt = delta > 0
    tang = np.where(filt)[0]
    if filt.sum() > 0:
        d1 = -(single_dot[filt]) + np.sqrt(delta[filt])
        sol1 = o[:, filt] + d1*u[:, filt]
        d2 = -(single_dot[filt]) - np.sqrt(delta[filt])
        sol2 = o[:, filt] + d2*u[:, filt]
        v1 = ((o[:, filt]-sol1)**2).sum()
        v2 = ((o[:, filt]-sol2)**2).sum()
        if v2 > v1:
            solution[1, ..., tang] = sol1.T[..., None]
        else:
            solution[1, ..., tang] = sol2.T[..., None]

    return solution


def normalize(v, axis=0):
    norm = np.linalg.norm(v, axis=axis)
    solution = v/norm
    solution[:, norm == 0] = v[:, norm == 0]
    return solution

def compute_line_3d(v, t, axis=0):
    """
    Generates a line given two cartesian points
    """
    v = np.array(v)
    t = np.array(t)
    o = v[...]
    u = normalize(t-v, axis=axis)
    return o, u
    
def compute_path_length_3d(R, altitudes, viewer, tangent,
                           coordinates='cartesian'):
    """
    Given a viewing and tangent vector, computes the path length for a sphere
    """

    _viewer = viewer[...]
    _tangent = tangent[...]

    if not hasattr(altitudes, '__len__'):
        altitudes = np.array([altitudes])
    else:
        altitudes = np.array(altitudes)

    if not isinstance(coordinates, (list, tuple,)):

        coordinates = [coordinates, coordinates]

    if coordinates[0] in ('spherical'):
        _viewer = sphere_to_cartesian(R, _viewer)
    if coordinates[1] in ('spherical'):
        _tangent = sphere_to_cartesian(R, _tangent)

    if len(_viewer.shape) == 1:
        _viewer = _viewer.reshape(-1, 1)
        _tangent = _viewer.reshape(-1, 1)

    o, u = compute_line_3d(_viewer,
                           _tangent)
    intersections = compute_intersection_3d(R, altitudes, u, o)

    if intersections is not None:

        distances = (np.linalg.norm(intersections[1] - intersections[0],
                                    axis=0))
        filt = np.isfinite(distances)
        all_distances = []

        for i in range(_viewer.shape[1]):
            layer_filt = filt[:, i]
            good_indices = np.where(layer_filt)[0]
            dists = distances[layer_filt, i]
            final_distances = np.zeros_like(dists)
            final_distances[0] = dists[0]
            final_distances[1:] = dists[1:]-dists[:-1]

            all_distances.append((good_indices, final_distances))
        return all_distances
    else:
        return None

In [ ]:
def parallel_vector(R, alt, max_alt=1e5):
    """
    Generates a viewing and tangent vectors
    parallel to the surface of a sphere
    """

    if not hasattr(alt, '__len__'):
        alt = np.array([alt])
    viewer = np.zeros(shape=(3, len(alt)))
    tangent = np.zeros_like(viewer)
    viewer[0] = -(R+max_alt*2)
    viewer[1] = R+alt
    tangent[1] = R+alt

    return viewer, tangent


def compute_path_length_planet(
    altitudes: npt.NDArray[np.float64],
    viewer: npt.NDArray[np.float64],
    tangent: npt.NDArray[np.float64],
    _planet,
    vector_coord_sys: t.Optional[t.Literal["cartesian"]] = "cartesian",
):
    """Compute path length through atmosphere."""
    # from taurex.util.geometry import compute_path_length_3d

    result = compute_path_length_3d(
        _planet.fullRadius, altitudes, viewer, tangent, coordinates=vector_coord_sys
    )

    return result
    
def compute_path_length(_model, _planet) -> t.List[npt.NDArray[np.float64]]:
    """Compute path length for each layer, new method."""
    # from taurex.util.geometry import parallel_vector

    altitude_boundaries = _model.altitude_boundaries
    radius = _planet.fullRadius

    # Generate our line of sight paths
    viewer, tangent = parallel_vector(
        radius, _model.altitude_profile + _model.deltaz / 2, altitude_boundaries.max()
    )
    print('altitude_boundaries', type(altitude_boundaries))
    print('radius', type(radius))
    print('_model.altitude_profile', type(_model.altitude_profile))
    print('_model.deltaz', type(_model.deltaz))

    path_lengths = compute_path_length_planet(
        altitude_boundaries, viewer, tangent, _planet
    )
    print('num path lengths', len(path_lengths))

    return [l for _, l in path_lengths]

def compute_absorption(
    tau: npt.NDArray[np.float64], dz: npt.NDArray[np.float64], _model, _planet, _star
) -> t.Tuple[npt.NDArray[np.float64], npt.NDArray[np.float64]]:
    """Compute final absorption and optical depth."""
    tau = np.exp(-tau)
    ap = _model.altitudeProfile[:, None]
    pradius = _planet.fullRadius
    sradius = _star.radius
    _dz = dz[:, None]

    print('altitudeProfile', type(ap))
    print('pradius', type(pradius))
    print('sradius', type(sradius))

    integral = np.sum((pradius + ap) * (1.0 - tau) * _dz * 2.0, axis=0)
    return ((pradius**2.0) + integral) / (sradius**2), tau

In [ ]:
def contribute_tau_numpy(
    startk: int,
    endk: int,
    density_offset: int,
    sigma: npt.NDArray[np.float64],
    density: npt.NDArray[np.float64],
    path: npt.NDArray[np.float64],
    # nlayers: int,
    # ngrid: int,
    layer: int,
    tau: npt.NDArray[np.float64],
) -> npt.NDArray[np.float64]:
    """Generic cross-section integration function for tau

    This has the form:

    .. math::

        \\tau_{\\lambda}(z) = \\int_{z_{0}}^{z_{1}} \\sigma(z') \\rho(z') dz',

    where :math:`z` is the layer, :math:`z_0` and :math:`z_1` are ``startK``
    and ``endK`` respectively. :math:`\\sigma` is the weighted
    cross-section ``sigma``. :math:`rho` is the ``density`` and
    :math:`dz'` is the integration path length ``path``


    Parameters
    ----------
    startK: int
        starting layer in integration

    endK: int
        last layer in integration

    density_offset: int
        Which part of the density profile to start from

    sigma: :obj:`array`
        cross-section

    density: array_like
        density profile of atmosphere

    path: array_like
        path-length or altitude gradient

    nlayers: int
        Total number of layers (unused)

    ngrid: int
        total number of grid points

    layer: int
        Which layer we currently on

    Returns
    -------
    tau : array_like
        optical depth (well almost you still need to do ``exp(-tau)`` yourself)

    """
    _path = path[startk:endk, None]
    _density = density[startk + density_offset : endk + density_offset, None]
    _sigma = sigma[startk + layer : endk + layer, :]

    tau[layer, :] += np.sum(_sigma * _path * _density, axis=0)

    return tau

In [ ]:
def path_integral(
    wngrid: npt.NDArray[np.float64], _model, _planet, _star, _contribution_list
) -> t.Tuple[npt.NDArray[np.float64], npt.NDArray[np.float64]]:
    """Compute path integral.

    Calculates the absorption and optical depth for each layer assuming
    hemispherical geometry.

    """
    dz = _model.deltaz

    total_layers = _model.nLayers

    wngrid_size = wngrid.shape[0]

    density_profile = _model.densityProfile

    path_length = compute_path_length(_model, _planet)
    print('path length shapes', [pl.shape for pl in path_length])

    # if self.new_method:
    #     path_length = self.compute_path_length()
    # else:
    #     path_length = self.compute_path_length_old(dz)
    # self.path_length = path_length

    tau = np.zeros(shape=(total_layers, wngrid_size), dtype=np.float64)

    for layer in range(total_layers):
        # self.debug("Computing layer %s", layer)
        dl = path_length[layer]

        end_k = total_layers - layer

        for contrib in _contribution_list:
            if tau[layer].min() > 10:
                break
            # self.debug("Adding contribution from %s", contrib.name)
            # contrib.contribute(
            #     _model, 0, end_k, layer, layer, density_profile, tau, path_length=dl
            # )

            # this returns tau, but tau is modified in place
            # contribute_tau_numpy(0, end_k, layer, contrib.sigma_xsec, density_profile, dl, layer, tau)
            tau = contribute_tau_numpy(0, end_k, layer, contrib.sigma_xsec, density_profile, dl, layer, tau)
    print('contrib.sigma_xsec', type(contrib.sigma_xsec))
    print('tau', tau.shape)
    

    absorption, tau = compute_absorption(tau, dz, _model, _planet, _star)
    return absorption, tau

In [ ]:
absorption, tau = path_integral(wn_grid, tm, planet, star, tm.contribution_list)

In [ ]:
absorption

## Clean

In [ ]:
def multi_dot(a, b):
    return np.sum(a*b, axis=0)
    
def compute_intersection_3d(R, h, u, o, c=np.zeros(3), allow_single=False):
    # Computes line-sphere intersection for a range of radius offsets
    # Will detect if path crosses R and cutoff the point
    h = np.array(h)
    
    if len(u.shape) == 1:
        u = u.reshape(-1, 1)
        o = o.reshape(-1, 1)
    
    single_dot = multi_dot(u, o)
    dot_res = (single_dot)**2 - (o**2).sum(axis=0)
    delta = dot_res + (R+h[..., None])**2
    solution = np.zeros(shape=(2, 3, h.shape[0], u.shape[1]))
    sol1 = np.zeros(shape=(3, h.shape[0], u.shape[1]))
    sol2 = np.zeros(shape=(3, h.shape[0], u.shape[1]))
    solution[...] = np.nan
    filt = delta > 0
    if allow_single:
        filt = delta >= 0
    if filt.sum() == 0:
        return None
    
    with np.errstate(divide='ignore', invalid='ignore'):

        d1 = -(single_dot) + np.sqrt(delta)
        d1[d1 < 0] = 0.0
        sol1 = o[:, None, :] + d1*u[:, None]

        d2 = -(single_dot) - np.sqrt(delta)
        d2[d2 < 0] = 0.0
        sol2 = o[:, None, :] + d2*u[:, None]
        v1 = np.sum((o[:, None, :]-sol1)**2, axis=0)
        v2 = np.sum((o[:, None, :]-sol2)**2, axis=0)

    max_filter = v2 > v1
    a_filter = max_filter
    b_filter = ~max_filter
    if a_filter.sum() > 0:
        solution[0, :, a_filter] = sol1[:, a_filter].T
        solution[1, :, a_filter] = sol2[:, a_filter].T
    if b_filter.sum() > 0:
        solution[0, :, b_filter] = sol2[:, b_filter].T

        solution[1, :, b_filter] = sol1[:, b_filter].T
    
    # Detect planet crossings
    delta = dot_res + (R)**2

    filt = delta > 0
    tang = np.where(filt)[0]
    if filt.sum() > 0:
        d1 = -(single_dot[filt]) + np.sqrt(delta[filt])
        sol1 = o[:, filt] + d1*u[:, filt]
        d2 = -(single_dot[filt]) - np.sqrt(delta[filt])
        sol2 = o[:, filt] + d2*u[:, filt]
        v1 = ((o[:, filt]-sol1)**2).sum()
        v2 = ((o[:, filt]-sol2)**2).sum()
        if v2 > v1:
            solution[1, ..., tang] = sol1.T[..., None]
        else:
            solution[1, ..., tang] = sol2.T[..., None]

    return solution


def normalize(v, axis=0):
    norm = np.linalg.norm(v, axis=axis)
    solution = v/norm
    solution[:, norm == 0] = v[:, norm == 0]
    return solution

def compute_line_3d(v, t, axis=0):
    # Generates a line given two cartesian points
    v = np.array(v)
    t = np.array(t)
    o = v[...] # is this a copy?
    u = normalize(t-v, axis=axis)
    return o, u
    
def compute_path_length_3d(R, altitudes, viewer, tangent,
                           coordinates='cartesian'):
    # Given a viewing and tangent vector, computes the path length for a sphere

    _viewer = viewer[...]
    _tangent = tangent[...]

    altitudes = np.array(altitudes)

    if not isinstance(coordinates, (list, tuple,)):
        coordinates = [coordinates, coordinates]

    if coordinates[0] in ('spherical'):
        _viewer = sphere_to_cartesian(R, _viewer)
    if coordinates[1] in ('spherical'):
        _tangent = sphere_to_cartesian(R, _tangent)

    if len(_viewer.shape) == 1:
        _viewer = _viewer.reshape(-1, 1)
        _tangent = _viewer.reshape(-1, 1)

    o, u = compute_line_3d(_viewer, _tangent)
    intersections = compute_intersection_3d(R, altitudes, u, o)

    if intersections is not None:

        distances = (np.linalg.norm(intersections[1] - intersections[0],
                                    axis=0))
        filt = np.isfinite(distances)
        all_distances = []

        for i in range(_viewer.shape[1]):
            layer_filt = filt[:, i]
            good_indices = np.where(layer_filt)[0]
            dists = distances[layer_filt, i]
            final_distances = np.zeros_like(dists)
            final_distances[0] = dists[0]
            final_distances[1:] = dists[1:]-dists[:-1]

            all_distances.append((good_indices, final_distances))
        return all_distances
    else:
        print('no intersections')
        return None

In [ ]:
def parallel_vector(R, alt, max_alt=1e5):
    """
    Generates a viewing and tangent vectors
    parallel to the surface of a sphere
    """

    if not hasattr(alt, '__len__'):
        alt = np.array([alt]) # i.e. convert to array if scalar
    viewer = np.zeros(shape=(3, len(alt)))
    tangent = np.zeros_like(viewer)
    viewer[0] = -(R+max_alt*2)
    viewer[1] = R+alt
    tangent[1] = R+alt

    return viewer, tangent


def compute_path_length_planet(
    altitudes: npt.NDArray[np.float64],
    viewer: npt.NDArray[np.float64],
    tangent: npt.NDArray[np.float64],
    _planet,
    vector_coord_sys: t.Optional[t.Literal["cartesian"]] = "cartesian",
):
    """Compute path length through atmosphere."""

    result = compute_path_length_3d(
        _planet.fullRadius, altitudes, viewer, tangent, coordinates=vector_coord_sys
    )

    return result
    
def compute_path_length(_model, _planet) -> t.List[npt.NDArray[np.float64]]:
    """Compute path length for each layer, new method."""

    altitude_boundaries = _model.altitude_boundaries
    print('altitude_boundaries', type(altitude_boundaries))
    radius = _planet.fullRadius

    # Generate our line of sight paths
    viewer, tangent = parallel_vector(
        radius, _model.altitude_profile + _model.deltaz / 2, altitude_boundaries.max()
    )

    path_lengths = compute_path_length_planet(
        altitude_boundaries, viewer, tangent, _planet
    )

    return [l for _, l in path_lengths]

def compute_absorption(
    tau: npt.NDArray[np.float64], dz: npt.NDArray[np.float64], _model, _planet, _star
) -> t.Tuple[npt.NDArray[np.float64], npt.NDArray[np.float64]]:
    """Compute final absorption and optical depth."""
    tau = np.exp(-tau)
    ap = _model.altitudeProfile[:, None]
    pradius = _planet.fullRadius
    sradius = _star.radius
    print('ap', type(ap))
    print('pradius', type(pradius))
    print('sradius', type(sradius))
    _dz = dz[:, None]

    integral = np.sum((pradius + ap) * (1.0 - tau) * _dz * 2.0, axis=0)
    return ((pradius**2.0) + integral) / (sradius**2), tau

def contribute_tau_numpy(
    startk: int,
    endk: int,
    density_offset: int,
    sigma: npt.NDArray[np.float64],
    density: npt.NDArray[np.float64],
    path: npt.NDArray[np.float64],
    # nlayers: int,
    # ngrid: int,
    layer: int,
    tau: npt.NDArray[np.float64],
) -> npt.NDArray[np.float64]:
    # Generic cross-section integration function for tau

    _path = path[startk:endk, None]
    _density = density[startk + density_offset : endk + density_offset, None]
    _sigma = sigma[startk + layer : endk + layer, :]

    tau[layer, :] += np.sum(_sigma * _path * _density, axis=0)

    return tau

def path_integral(
    wngrid: npt.NDArray[np.float64], _model, _planet, _star, _contribution_list
) -> t.Tuple[npt.NDArray[np.float64], npt.NDArray[np.float64]]:
    # Calculates the absorption and optical depth for each layer assuming
    # hemispherical geometry.
    dz = _model.deltaz
    print('dz', type(dz))

    total_layers = _model.nLayers
    print('total_layers', type(total_layers))

    wngrid_size = wngrid.shape[0]

    density_profile = _model.densityProfile
    print('density_profile', type(density_profile))

    path_length = compute_path_length(_model, _planet)

    tau = np.zeros(shape=(total_layers, wngrid_size), dtype=np.float64)

    for layer in range(total_layers):
        dl = path_length[layer]

        end_k = total_layers - layer

        for contrib in _contribution_list:
            if tau[layer].min() > 10:
                break
            tau = contribute_tau_numpy(0, end_k, layer, contrib.sigma_xsec, density_profile, dl, layer, tau)

    # just print last one
    print('contrib.sigma_xsec', type(contrib.sigma_xsec))

    absorption, tau = compute_absorption(tau, dz, _model, _planet, _star)
    return absorption, tau

In [ ]:
absorption, tau = path_integral(wn_grid, tm, planet, star, tm.contribution_list)

In [ ]:
_y = obin.bin_model((wn_grid, absorption, tau, None))[1]

plt.figure()
plt.errorbar(obs.wavelengthGrid, obs.spectrum, obs.errorBar, label='Obs')
plt.plot(obs.wavelengthGrid, _y, label='TM')
plt.legend()
plt.show()

# Jaxify

In [ ]:
import jax.numpy as jnp

In [ ]:
def multi_dot(a, b):
    return np.sum(a*b, axis=0)
    
def compute_intersection_3d(R, h, u, o, c=np.zeros(3), allow_single=False):
    # Computes line-sphere intersection for a range of radius offsets
    # Will detect if path crosses R and cutoff the point
    h = np.array(h)
    
    if len(u.shape) == 1:
        u = u.reshape(-1, 1)
        o = o.reshape(-1, 1)
    
    single_dot = multi_dot(u, o)
    dot_res = (single_dot)**2 - (o**2).sum(axis=0)
    delta = dot_res + (R+h[..., None])**2
    solution = np.zeros(shape=(2, 3, h.shape[0], u.shape[1]))
    sol1 = np.zeros(shape=(3, h.shape[0], u.shape[1]))
    sol2 = np.zeros(shape=(3, h.shape[0], u.shape[1]))
    solution[...] = np.nan
    filt = delta > 0
    if allow_single:
        filt = delta >= 0
    if filt.sum() == 0:
        return None
    
    with np.errstate(divide='ignore', invalid='ignore'):

        d1 = -(single_dot) + np.sqrt(delta)
        d1[d1 < 0] = 0.0
        sol1 = o[:, None, :] + d1*u[:, None]

        d2 = -(single_dot) - np.sqrt(delta)
        d2[d2 < 0] = 0.0
        sol2 = o[:, None, :] + d2*u[:, None]
        v1 = np.sum((o[:, None, :]-sol1)**2, axis=0)
        v2 = np.sum((o[:, None, :]-sol2)**2, axis=0)

    max_filter = v2 > v1
    a_filter = max_filter
    b_filter = ~max_filter
    if a_filter.sum() > 0:
        solution[0, :, a_filter] = sol1[:, a_filter].T
        solution[1, :, a_filter] = sol2[:, a_filter].T
    if b_filter.sum() > 0:
        solution[0, :, b_filter] = sol2[:, b_filter].T

        solution[1, :, b_filter] = sol1[:, b_filter].T
    
    # Detect planet crossings
    delta = dot_res + (R)**2

    filt = delta > 0
    tang = np.where(filt)[0]
    if filt.sum() > 0:
        d1 = -(single_dot[filt]) + np.sqrt(delta[filt])
        sol1 = o[:, filt] + d1*u[:, filt]
        d2 = -(single_dot[filt]) - np.sqrt(delta[filt])
        sol2 = o[:, filt] + d2*u[:, filt]
        v1 = ((o[:, filt]-sol1)**2).sum()
        v2 = ((o[:, filt]-sol2)**2).sum()
        if v2 > v1:
            solution[1, ..., tang] = sol1.T[..., None]
        else:
            solution[1, ..., tang] = sol2.T[..., None]

    return solution


def normalize(v, axis=0):
    norm = np.linalg.norm(v, axis=axis)
    solution = v/norm
    solution[:, norm == 0] = v[:, norm == 0]
    return solution

def compute_line_3d(v, t, axis=0):
    # Generates a line given two cartesian points
    v = np.array(v)
    t = np.array(t)
    o = v[...] # is this a copy?
    u = normalize(t-v, axis=axis)
    return o, u
    
def compute_path_length_3d(R, altitudes, viewer, tangent,
                           coordinates='cartesian'):
    # Given a viewing and tangent vector, computes the path length for a sphere

    _viewer = viewer[...]
    _tangent = tangent[...]

    altitudes = np.array(altitudes)

    if not isinstance(coordinates, (list, tuple,)):
        coordinates = [coordinates, coordinates]

    if coordinates[0] in ('spherical'):
        _viewer = sphere_to_cartesian(R, _viewer)
    if coordinates[1] in ('spherical'):
        _tangent = sphere_to_cartesian(R, _tangent)

    if len(_viewer.shape) == 1:
        _viewer = _viewer.reshape(-1, 1)
        _tangent = _viewer.reshape(-1, 1)

    o, u = compute_line_3d(_viewer, _tangent)
    intersections = compute_intersection_3d(R, altitudes, u, o)

    if intersections is not None:

        distances = (np.linalg.norm(intersections[1] - intersections[0],
                                    axis=0))
        filt = np.isfinite(distances)
        all_distances = []

        for i in range(_viewer.shape[1]):
            layer_filt = filt[:, i]
            good_indices = np.where(layer_filt)[0]
            dists = distances[layer_filt, i]
            final_distances = np.zeros_like(dists)
            final_distances[0] = dists[0]
            final_distances[1:] = dists[1:]-dists[:-1]

            all_distances.append((good_indices, final_distances))
        return all_distances
    else:
        print('no intersections')
        return None

In [ ]:
@jit
def compute_simple_path_lengths_jax(R, altitudes, n_layers):
    """
    Simplified JAX-friendly version of path length computation.
    Uses a more straightforward geometric approach.
    """
    altitudes = jnp.array(altitudes)
    path_lengths = []
    
    for layer in range(n_layers):
        # Impact parameter at this layer
        impact_radius = R + altitudes[layer]
        p_squared = impact_radius**2
        
        # Remaining layers above this one
        remaining_layers = n_layers - layer
        paths = jnp.zeros(remaining_layers)
        
        # Path through this layer
        r_this = R + altitudes[layer]
        path_this = jnp.sqrt(jnp.maximum(r_this**2 - p_squared, 0.0))
        paths = paths.at[0].set(path_this)
        
        # Paths through layers above
        if remaining_layers > 1:
            upper_altitudes = altitudes[layer + 1:layer + remaining_layers]
            r_upper = R + upper_altitudes
            upper_paths = jnp.sqrt(jnp.maximum(r_upper**2 - p_squared, 0.0))
            
            lower_altitudes = altitudes[layer:layer + remaining_layers - 1] 
            r_lower = R + lower_altitudes
            lower_paths = jnp.sqrt(jnp.maximum(r_lower**2 - p_squared, 0.0))
            
            paths = paths.at[1:].set(upper_paths - lower_paths)
        
        # Factor of 2 for both sides of planet
        path_lengths.append(paths * 2.0)
    
    return path_lengths

@jit
def parallel_vector_jax(R, alt, max_alt=1e5):
    """
    JAX version of parallel_vector.
    Generates viewing and tangent vectors parallel to the surface of a sphere.
    """
    alt = jnp.atleast_1d(alt)
    
    viewer = jnp.zeros((3, len(alt)))
    tangent = jnp.zeros_like(viewer)
    
    viewer = viewer.at[0].set(-(R + max_alt * 2))
    viewer = viewer.at[1].set(R + alt)
    tangent = tangent.at[1].set(R + alt)
    
    return viewer, tangent

@jit 
def perpendicular_vector_jax(R, max_alt):
    """
    JAX version of perpendicular_vector.
    Generates viewing and tangent vectors perpendicular to a sphere of radius R.
    """
    viewer = jnp.array([0, R + max_alt * 2, 0]).reshape(3, 1)
    tangent = jnp.array([0.0, 0, 0]).reshape(3, 1)
    return viewer, tangent